# Texcoco farm planner: MVP notebook

A what-if tool for rainfed farms in Texcoco (WOFOST point at CIMMYT El Batán, 19.53°N, 98.85°W). Change the controls to:

1. **Reality check:** compare WOFOST yields with what Texcoco farms report to SIAP.
2. **Farm explorer:** see income per hectare per year for each rotation and scenario, given prices and costs.
3. **Bad-year risk:** compare rotations against continuous maize in the same weather years.
4. **New WOFOST run:** simulate your own drought, fertilizer, frost or sowing scenario (about 25 s per rotation).
5. **Surrogate playground:** compare decision tree, random forest and XGBoost emulators of WOFOST maize yield.
6. **What to improve next.**

**Read before trusting a number.** WOFOST's crop parameters are mostly European, and section 1 shows its yields are several times what Texcoco farms report. Prices are SIAP nominal rural prices. **Production costs are placeholders.** Use the results to compare options with each other, not as yield forecasts or farm advice.

Run from the repo with the project environment: `.venv/bin/jupyter lab Notebooks/texcoco_farmer_mvp.ipynb`. Sections 1–3 and 5 use saved simulations and run offline. Section 4 needs internet the first time.

In [ ]:
%matplotlib inline
import os
import re
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "crop_models").is_dir())
os.chdir(ROOT)  # data paths and `src.` imports are relative to the repo root
sys.path.insert(0, str(ROOT))

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from sklearn.tree import export_text

from src.crop_models import farm_tool as ft
from src.crop_models.rotation_experiments import SCENARIOS, TARGET
from src.crop_models.rotation_risk import compare_rotations
from src.crop_models.tree_surrogate import benchmark, build_features

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
STYLE = {"description_width": "170px"}
LAYOUT = w.Layout(width="420px")
CROPS = ["maize", "barley", "fababean"]
ROTATIONS = ["maize", "maize-barley", "maize-fababean", "maize-barley-fababean"]

seasons = ft.load_scenarios()
siap = ft.siap_texcoco()
print(f"{seasons['scenario'].nunique()} scenarios, {len(seasons):,} simulated seasons, "
      f"years {seasons['year'].min()}-{seasons['year'].max()} (first year dropped as spin-up)")
print(f"SIAP Texcoco observations: {siap['year'].min()}-{siap['year'].max()}, crops {sorted(siap['crop'].unique())}")

## 1. Reality check: WOFOST vs what Texcoco farms report

SIAP publishes sown, harvested and lost (*siniestrada*) area and production for the whole municipality. **Yield per sown hectare** counts lost fields as zero, like a failed WOFOST season, so it is the fairer comparison.

* Simulated maize is weighted by SIAP crop areas: almost all Texcoco maize follows maize, so continuous maize counts most (`ft.simulated_like_siap`).
* `factor` = observed mean ÷ simulated mean. The explorer can multiply WOFOST yields by it ("farm-calibrated").
* `correlation` is the year-to-year agreement. A low value means WOFOST does not yet get the good and bad years right at this point. Calibrated yields fix the *level*, not the *pattern*.

In [ ]:
gap, paired = ft.farm_gap(seasons, siap)
display(gap)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for ax, crop in zip(axes, CROPS):
    d = paired[paired["crop"] == crop]
    ax.plot(d["year"], d["simulated_t_ha"], "o-", label="WOFOST baseline (all rotations)")
    ax.plot(d["year"], d["observed_t_ha"], "s-", label="SIAP, per sown ha")
    ax.plot(d["year"], d["simulated_t_ha"] * gap.loc[crop, "factor"], ":", color="gray", label="WOFOST x factor")
    ax.set(title=f"{crop}: factor {gap.loc[crop, 'factor']:.2f}, r = {gap.loc[crop, 'correlation']:.2f}",
           ylabel="t/ha")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

base = seasons[seasons["scenario"] == "baseline"]
low = base[TARGET] < 0.2 * base.groupby("crop")[TARGET].transform("mean")
display(pd.DataFrame({
    "WOFOST seasons below 20% of crop mean": low.groupby(base["crop"]).mean(),
    "SIAP share of sown area lost": siap.groupby("crop")["lost_share"].mean(),
}).rename_axis("crop"))

## 2. Farm explorer: income per hectare per year

Each rotation runs all its phases every year, as in a field trial. The **annual value of a rotation is the phase average**, which is what one hectare split across the phases produces that year. Over the years, it is also the average of a single field.

* Net = yield × price − (other costs + N fertilizer kg × N price). Fertilizer doses come from each scenario.
* Prices default to the median SIAP rural price for 2023–2025 (nominal MXN/t).
* **Other costs are placeholders** (seed, land preparation, labor and harvest, excluding fertilizer). Replace them with real figures.
* The heatmap ranks every scenario and rotation using the same prices and costs.

In [ ]:
prices = ft.default_prices(siap)
PLACEHOLDER_COSTS = {"maize": 14000, "barley": 9000, "fababean": 11000}  # MXN/ha, not sourced: replace

scenario_w = w.Dropdown(options=sorted(seasons["scenario"].unique()), value="baseline", description="Scenario",
                        style=STYLE, layout=LAYOUT)
yields_w = w.ToggleButtons(options=[("Farm-calibrated (SIAP level)", "calibrated"), ("WOFOST as simulated", "raw")],
                           description="Yields", style=STYLE)
n_price_w = w.FloatSlider(value=25, min=0, max=80, step=1, description="N price (MXN/kg N)", style=STYLE, layout=LAYOUT)
fail_w = w.FloatSlider(value=1.0, min=0, max=4, step=0.1, description="Failed harvest below (t/ha)", style=STYLE,
                       layout=LAYOUT)
price_w = {c: w.FloatText(value=round(prices[c], -1), step=100, description=f"{c} price (MXN/t)", style=STYLE,
                          layout=LAYOUT) for c in CROPS}
cost_w = {c: w.FloatText(value=PLACEHOLDER_COSTS[c], step=500, description=f"{c} other costs (MXN/ha)",
                         style=STYLE, layout=LAYOUT) for c in CROPS}


def yield_factors(mode):
    return gap["factor"].to_dict() if mode == "calibrated" else None


def explore(scenario, yields, n_price, failure, **money):
    prices = {c: money[f"price_{c}"] for c in CROPS}
    costs = {c: money[f"cost_{c}"] for c in CROPS}
    _, annual, summary, crops = ft.rotation_economics(seasons, prices, costs, n_price, yield_factors(yields), failure)
    one = summary[summary["scenario"] == scenario].set_index("rotation").reindex(ROTATIONS).dropna(how="all")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1.4, 1.3]})
    ax = axes[0]
    err = [one["mean_net_mxn_ha"] - one["p10_net_mxn_ha"], one["p90_net_mxn_ha"] - one["mean_net_mxn_ha"]]
    ax.bar(one.index, one["mean_net_mxn_ha"], yerr=err, capsize=4, color="#4c78a8")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{scenario}: mean net, bars = 10th-90th pct", ylabel="MXN/ha/year")
    ax.tick_params(axis="x", rotation=25)

    ax = axes[1]
    yearly = annual[annual["scenario"] == scenario].pivot(index="year", columns="rotation", values="net_mxn_ha")
    yearly.reindex(columns=[r for r in ROTATIONS if r in yearly]).plot(ax=ax, marker="o", markersize=3)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title="Net income by year (same weather for every rotation)", ylabel="MXN/ha", xlabel="")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(fontsize=8)

    ax = axes[2]
    grid = summary.pivot(index="scenario", columns="rotation", values="mean_net_mxn_ha")
    grid = grid.reindex(columns=[r for r in ROTATIONS if r in grid])
    limit = np.nanmax(np.abs(grid.to_numpy())) or 1
    ax.imshow(grid, cmap="RdYlGn", norm=TwoSlopeNorm(0, -limit, limit), aspect="auto")
    ax.set_xticks(range(grid.shape[1]), [c.replace("-", "-\n") for c in grid.columns], fontsize=7)
    ax.set_yticks(range(grid.shape[0]), grid.index, fontsize=7)
    for (i, j), value in np.ndenumerate(grid.to_numpy()):
        if np.isfinite(value):
            ax.text(j, i, f"{value / 1000:.1f}k", ha="center", va="center", fontsize=7)
    ax.set_title("Mean net MXN/ha/year, all scenarios")
    fig.tight_layout()
    plt.show()

    display(one.drop(columns="scenario").style.format("{:,.0f}").format("{:.0%}", subset=["loss_year_share"])
            .set_caption("loss_year_share = share of years with negative net income"))
    display(crops[crops["scenario"] == scenario].drop(columns="scenario").set_index(["rotation", "crop"])
            .style.format("{:.2f}").format("{:.0%}", subset=["failed_share"]).format("{:.0f}", subset=["seasons"]))


controls = {"scenario": scenario_w, "yields": yields_w, "n_price": n_price_w, "failure": fail_w,
            **{f"price_{c}": price_w[c] for c in CROPS}, **{f"cost_{c}": cost_w[c] for c in CROPS}}
display(w.VBox([
    w.HBox([w.VBox([scenario_w, n_price_w, fail_w]), w.VBox(list(price_w.values())), w.VBox(list(cost_w.values()))]),
    yields_w,
    w.interactive_output(explore, controls),
]))

## 3. Bad-year risk: maize in each rotation vs a reference

This section uses `rotation_risk.compare_rotations`. It pairs the same weather years across rotations and bootstraps them to find the **maize advantage** and its 95% interval. It also samples runs of future maize harvests to estimate how likely at least one failed harvest is.

The intervals cover year-to-year weather under fixed WOFOST assumptions, not model or calibration error. Harvest counts are maize harvests, so a two-year rotation needs twice the calendar time to produce the same number.

In [ ]:
risk_scenario_w = w.Dropdown(options=sorted(seasons["scenario"].unique()), value="baseline", description="Scenario",
                             style=STYLE, layout=LAYOUT)
reference_w = w.Dropdown(options=ROTATIONS, value="maize", description="Reference rotation", style=STYLE,
                         layout=LAYOUT)
harvests_w = w.IntSlider(value=10, min=1, max=30, description="Maize harvests sampled", style=STYLE, layout=LAYOUT)


def risk(scenario, reference, harvests, yields, failure):
    data = seasons[seasons["scenario"] == scenario].copy()
    if reference not in set(data["rotation"]):
        print(f"{scenario} has no {reference} rotation; pick another reference.")
        return
    if yields == "calibrated":
        data[TARGET] *= data["crop"].map(gap["factor"])
    effects, risks = compare_rotations(data, "maize", reference, draws=2000, harvests=harvests,
                                       failure_threshold=failure)
    effects = effects.set_index("rotation").reindex([r for r in ROTATIONS if r in set(effects["rotation"])])

    fig, ax = plt.subplots(figsize=(7, 3.2))
    low = effects["mean_advantage_t_ha"] - effects["advantage_ci_low_t_ha"]
    high = effects["advantage_ci_high_t_ha"] - effects["mean_advantage_t_ha"]
    ax.errorbar(effects.index, effects["mean_advantage_t_ha"], yerr=[low, high], fmt="o", capsize=5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(ylabel="t/ha", title=f"Maize advantage vs {reference} ({scenario}, {yields} yields), 95% interval")
    fig.tight_layout()
    plt.show()
    display(risks.set_index("rotation")[["paired_years", "mean_yield_t_ha", "mean_yield_p05_t_ha",
                                         "mean_yield_p95_t_ha", "failed_harvest_fraction",
                                         "prob_any_failed_harvest", "prob_mean_below_reference"]])


display(w.VBox([w.HBox([risk_scenario_w, reference_w, harvests_w]),
                w.HTML("Uses the <b>Yields</b> mode and <b>failed harvest</b> threshold from section 2."),
                w.interactive_output(risk, {"scenario": risk_scenario_w, "reference": reference_w,
                                            "harvests": harvests_w, "yields": yields_w, "failure": fail_w})]))

## 4. Run a new WOFOST scenario

This runs the full WOFOST rotation simulator at the Texcoco point for 2005–2025 and saves the result to `data/exploration/notebook_runs/<name>.csv` (git-ignored). The new scenario then appears in the dropdowns in sections 2 and 3, and on the next start-up. Each rotation takes about 25 s.

* **Rain factor** multiplies rain, either all year or only in the selected months (for example, July–August for a stronger *canícula*).
* **Fertilizer scale** multiplies every crop's N plan (maize 160 kg N/ha at 1.0).
* **Frost offset** is added to night temperatures; −2 means valley nights 2 °C colder than the weather grid.

In [ ]:
name_w = w.Text(value="my_scenario", description="Scenario name", style=STYLE, layout=LAYOUT)
rotations_w = w.SelectMultiple(options=ROTATIONS, value=ROTATIONS, rows=4, description="Rotations", style=STYLE,
                               layout=LAYOUT)
rain_w = w.FloatSlider(value=1.0, min=0.3, max=1.3, step=0.05, description="Rain factor", style=STYLE, layout=LAYOUT)
months_w = w.SelectMultiple(options=[(m, i) for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1)], rows=6,
    description="Only these months (none = all)", style=STYLE, layout=LAYOUT)
n_scale_w = w.FloatSlider(value=1.0, min=0, max=2, step=0.1, description="Fertilizer scale", style=STYLE, layout=LAYOUT)
frost_w = w.FloatSlider(value=0.0, min=-4, max=2, step=0.5, description="Frost offset (°C)", style=STYLE, layout=LAYOUT)
sowing_w = w.ToggleButtons(options=[("When rains start", "onset"), ("Fixed date", "fixed")], description="Sowing",
                           style=STYLE)
run_w = w.Button(description="Run WOFOST", button_style="primary", icon="play")
log_w = w.Output()


def run_clicked(_):
    global seasons
    with log_w:
        log_w.clear_output()
        name = re.sub(r"[^a-z0-9_]+", "_", name_w.value.lower()).strip("_")
        if not name or name in SCENARIOS:
            print("Choose a new name (letters, digits, _), not one of the saved suites.")
            return
        if not rotations_w.value:
            print("Select at least one rotation.")
            return
        print(f"Running WOFOST for '{name}' ({len(rotations_w.value)} rotation(s), ~25 s each)...")
        run_w.disabled = True
        try:
            new = ft.run_scenario(name, list(rotations_w.value), rain_w.value, sorted(months_w.value) or None,
                                  frost_w.value, n_scale_w.value, sowing_w.value)
        finally:
            run_w.disabled = False
        seasons = pd.concat([seasons[seasons["scenario"] != name], new], ignore_index=True)
        for dropdown in (scenario_w, risk_scenario_w):
            dropdown.options = sorted(seasons["scenario"].unique())
        scenario_w.value = name
        print(f"Done: {len(new)} seasons. '{name}' is selected in section 2 and available in section 3.")
        display(new[new["crop"] == "maize"].groupby("rotation")[TARGET].describe()[["mean", "min", "max"]])


run_w.on_click(run_clicked)
display(w.VBox([w.HBox([w.VBox([name_w, rotations_w, sowing_w]), w.VBox([rain_w, months_w]),
                        w.VBox([n_scale_w, frost_w, run_w])]), log_w]))

## 5. Surrogate playground: decision tree, random forest and XGBoost

These models emulate **WOFOST maize yield** from information available before the season: rotation, previous crop, starting soil water and mineral N, fertilizer, and the scenario settings. They train on 2006–2021 and are scored on 2022–2025 across every loaded scenario, using the same settings as `tree_surrogate.py`. They answer instantly where WOFOST takes about 25 s per rotation, but they do not know the coming season's weather. Expect errors of 2–3 t/ha.

The scenarios simulate maize at only three fertilizer doses: 0, 80 and 160 kg N/ha. Tree models therefore produce **step functions** between these doses and stay flat above 160 (shaded). Read the fertilizer curve as "which side of a simulated dose", not as a dose-response. Adding WOFOST runs at intermediate doses through section 4 is the fix.

Run the next cell again after adding scenarios in section 4 to retrain the models on them.

In [ ]:
metrics, holdout_predictions, _, models, selected = benchmark(seasons, "maize", "planning")
display(metrics[metrics["split"].isin(["cv_pooled", "holdout"])]
        .pivot(index="model", columns="split", values=["rmse_t_ha", "r2"]).round(3))
print(f"Selected by development-year cross-validation: {selected}")

importance = pd.DataFrame({
    name: pd.Series(m["model"].feature_importances_, m["features"].get_feature_names_out())
    for name, m in models.items() if hasattr(m["model"], "feature_importances_")})
importance.sort_values("xgboost" if "xgboost" in importance else importance.columns[-1]).plot.barh(figsize=(8, 4.5))
plt.title("Feature importance (impurity/gain based; correlated features share credit)")
plt.tight_layout()
plt.show()

In [ ]:
maize = seasons[seasons["crop"] == "maize"]
previous = maize.groupby("rotation")["previous_crop"].agg(lambda x: x.mode()[0]).to_dict()
s_rotation_w = w.Dropdown(options=ROTATIONS, value="maize-fababean", description="Rotation", style=STYLE, layout=LAYOUT)
s_water_w = w.FloatSlider(value=maize["soil_water_at_start_mm"].median(), min=100, max=400, step=5,
                          description="Soil water at start (mm)", style=STYLE, layout=LAYOUT)
s_n_w = w.FloatSlider(value=maize["mineral_n_at_start_kg_ha"].median(), min=0, max=130, step=1,
                      description="Mineral N at start (kg/ha)", style=STYLE, layout=LAYOUT)
s_rain_w = w.FloatSlider(value=1.0, min=0.5, max=1.2, step=0.05, description="Rain factor", style=STYLE, layout=LAYOUT)
s_months_w = w.Dropdown(options=[("All year", "all"), ("July-August only", "7,8")], description="Rain change in",
                        style=STYLE, layout=LAYOUT)
s_nscale_w = w.FloatSlider(value=1.0, min=0, max=1.5, step=0.05, description="Fertilizer scale", style=STYLE,
                           layout=LAYOUT)
s_frost_w = w.FloatSlider(value=0.0, min=-3, max=1, step=0.5, description="Frost offset (°C)", style=STYLE,
                          layout=LAYOUT)
s_sowing_w = w.Dropdown(options=["onset", "fixed"], description="Sowing", style=STYLE, layout=LAYOUT)


def feature_rows(rotation, water, n, rain, months, n_scale, frost, sowing):
    """Planning-feature rows for maize; n_scale may be an array (fertilizer is 160 kg N/ha x scale)."""
    n_scale = np.atleast_1d(n_scale)
    rows = pd.DataFrame({"rotation": rotation, "previous_crop": previous[rotation], "sowing_mode": sowing,
                         "rain_months": months, "n_fertilizer_kg_ha": 160 * n_scale, "soil_water_at_start_mm": water,
                         "mineral_n_at_start_kg_ha": n, "rain_factor": rain, "n_scale": n_scale,
                         "frost_tmin_offset": frost})
    return build_features(rows)[0]


def playground(rotation, water, n, rain, months, n_scale, frost, sowing):
    args = (rotation, water, n, rain, months)
    point = feature_rows(*args, n_scale, frost, sowing)
    sweep = np.linspace(0, 1.5, 31)
    grid = feature_rows(*args, sweep, frost, sowing)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    predictions = {name: max(0, m.predict(point)[0]) for name, m in models.items()}
    # "mean" is the benchmark baseline: one fixed number, the average maize yield of every training season
    # (all scenarios, rotations and years 2006-2021), so it does not react to the sliders.
    labels = ["average of all\ntraining seasons\n(ignores sliders)" if name == "mean" else name for name in predictions]
    axes[0].bar(labels, predictions.values(), color=["#bbb", "#72b7b2", "#54a24b", "#e45756"])
    for i, value in enumerate(predictions.values()):
        axes[0].text(i, value, f"{value:.1f}\n(x{gap.loc['maize', 'factor']:.2f} = "
                     f"{value * gap.loc['maize', 'factor']:.1f})", ha="center", va="bottom", fontsize=8)
    axes[0].set(ylabel="WOFOST-equivalent maize t/ha", title="Predicted yield (farm-calibrated in brackets)")
    axes[0].set_ylim(0, max(predictions.values()) * 1.35 + 0.5)
    for name, m in models.items():
        if name != "mean":
            axes[1].plot(sweep * 160, np.maximum(0, m.predict(grid)), label=name)
    observed = maize[maize["rotation"] == rotation].groupby("scenario").agg(
        n=("n_fertilizer_kg_ha", "first"), y=(TARGET, "mean"))
    axes[1].scatter(observed["n"], observed["y"], color="black", s=18, zorder=3, label="WOFOST scenario means")
    axes[1].axvline(160 * n_scale, color="gray", linestyle=":")
    axes[1].axvspan(160, 240, color="0.93", zorder=0, label="beyond simulated doses")
    axes[1].set(xlabel="Maize N fertilizer (kg N/ha)", ylabel="t/ha", title=f"Fertilizer response, {rotation}")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()


display(w.VBox([w.HBox([w.VBox([s_rotation_w, s_water_w, s_n_w, s_sowing_w]),
                        w.VBox([s_rain_w, s_months_w, s_nscale_w, s_frost_w])]),
                w.interactive_output(playground, {"rotation": s_rotation_w, "water": s_water_w, "n": s_n_w,
                                                  "rain": s_rain_w, "months": s_months_w, "n_scale": s_nscale_w,
                                                  "frost": s_frost_w, "sowing": s_sowing_w})]))

In [ ]:
# The decision tree is small enough to read as rules.
tree = models["decision_tree"]
print(export_text(tree["model"], feature_names=list(tree["features"].get_feature_names_out()), decimals=1))

## 6. What to improve next

Findings from building this MVP, roughly in order of value for a farmer-facing tool:

1. **Use the calibrated runs.** Section 1 shows the default level is several times the SIAP level for every crop. A calibration now exists (`src/crop_models/calibration.py`, explained step by step in `docs/technical_report/calibration.pdf`); its runs in `data/rotations_calibrated/` match SIAP's level, so the factors above become about 1. It still does not beat the SIAP average year by year, and once calibrated, maize no longer benefits from rotations. Switch this notebook to the calibrated runs, and note that 2023's highest observed maize yield is a near-failure for continuous maize in the default model (other rotations do well).
2. **Point vs municipality.** One NASA POWER grid cell (about 50 km) drives a municipality-wide average. Add a few points across Texcoco or a finer rain product (CHIRPS), and compare them with SIAP's lost area.
3. **Replace placeholder costs** with published cost-of-production tables for rainfed maize, barley and faba bean in Estado de México, such as FIRA *costos de producción*. Add a price-risk scenario: maize prices doubled between 2020 and 2022.
4. **Pre-season information.** The planning surrogate ignores the coming season's weather, so it only learns scenario settings and starting soil state. A seasonal or ENSO outlook feature is the natural next input. Validate it on fresh years.
5. **Faster what-ifs.** Run WOFOST over a management grid (fertilizer, sowing date, rotation) × all weather years once. The notebook could then interpolate instantly instead of waiting about 25 s per rotation.
6. **Toward a farmer tool:** use Spanish, fewer controls (field size, previous crop, fertilizer kg, sowing week), and two outputs: expected income and the chance of a bad year.